# Data preprocessing ? Custom CNN
Reproducible 70/15/15 stratified split; source files are never modified.

In [1]:
from pathlib import Path
import csv, json, random
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
RANDOM_SEED=42; ROOT=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd(); DATASET=ROOT/"PlantVillage"; OUT=ROOT/"results/preprocessing"; OUT.mkdir(parents=True,exist_ok=True)
EXTS={".jpg",".jpeg",".png",".bmp",".gif",".tif",".tiff"}; rows=[]
for d in sorted(p for p in DATASET.iterdir() if p.is_dir()):
 for p in sorted(d.rglob("*")):
  if p.is_file() and p.suffix.lower() in EXTS: rows.append((str(p.resolve()),d.name))
classes=sorted({x[1] for x in rows}); ci={c:i for i,c in enumerate(classes)}; labels=[ci[x[1]] for x in rows]
tr,rem=train_test_split(range(len(rows)),test_size=.30,stratify=labels,random_state=RANDOM_SEED); va,te=train_test_split(rem,test_size=.50,stratify=[labels[i] for i in rem],random_state=RANDOM_SEED); split={i:"train" for i in tr}|{i:"validation" for i in va}|{i:"test" for i in te}; allrows=[[rows[i][0],rows[i][1],labels[i],split[i]] for i in range(len(rows))]
def w(name,data,head):
 with open(OUT/name,"w",newline="",encoding="utf-8") as f: csv.writer(f).writerows([head,*data])
w("dataset_split.csv",allrows,["filepath","class_name","class_index","split"]); [w(f"{s}_metadata.csv",[r for r in allrows if r[3]==s],["filepath","class_name","class_index","split"]) for s in ["train","validation","test"]]
summary=[[c,sum(r[1]==c for r in allrows),sum(r[1]==c and r[3]=="train" for r in allrows),sum(r[1]==c and r[3]=="validation" for r in allrows),sum(r[1]==c and r[3]=="test" for r in allrows)] for c in classes]; w("class_split_counts.csv",summary,["class_name","total","train","validation","test"]); w("split_summary.csv",[["total",len(allrows)],["train",len(tr)],["validation",len(va)],["test",len(te)]],["split","count"]); weights=compute_class_weight("balanced",classes=np.arange(len(classes)),y=[labels[i] for i in tr]); w("class_weights.csv",[[c,i,float(weights[i])] for i,c in enumerate(classes)],["class_name","class_index","weight"]); json.dump({"image_size":[224,224],"channels":3,"normalization":"pixel/255","augmentation_training_only":True,"seed":42},open(OUT/"preprocessing_config.json","w"),indent=2); print(len(classes),len(rows),len(tr),len(va),len(te),[len(set(allrows[i][0] for i in a)&set(allrows[j][0] for j in b)) for a,b in [(tr,va),(tr,te),(va,te)]])

15 4172 2920 626 626 [0, 0, 0]
